# Geotechnical Feature Engineering
## Phase 3: Advanced Behavioral Feature Derivation

**Objective:** Enhance predictive signal through domain-informed feature construction.

**Key Features:**
1. **Liquidity Index (LI)** - Soil consistency indicator
2. **Synthetic CPT Resistance** - SPT-to-CPT correlation
3. **Derived indices** for improved model performance

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Environment initialized")

---
## 1. Load Cleaned Dataset

In [ ]:
# Load cleaned data from Phase 2
df = pd.read_csv('/home/claude/geotechnical_cleaned.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nAvailable Columns:")
print(list(df.columns))
print(f"\n{'='*60}")

---
## 2. Liquidity Index (LI)

### Formula: $LI = \frac{W_n - PL}{LL - PL}$

**Physical Interpretation:**
- LI < 0: Soil is drier than plastic limit (brittle)
- LI = 0: At plastic limit
- 0 < LI < 1: Plastic state (workable)
- LI = 1: At liquid limit
- LI > 1: Liquid state (flows)

**Engineering Significance:**
- Strong correlation with undrained shear strength
- Critical for slope stability and foundation design

In [ ]:
def calculate_liquidity_index(row):
    """
    Calculate Liquidity Index (LI) from moisture content and Atterberg limits.
    
    LI = (Wn - PL) / (LL - PL)
    
    Returns:
        float: Liquidity Index value, or NaN if calculation is impossible
    """
    Wn = row.get('Moisture_Content', np.nan)
    PL = row.get('PL', np.nan)
    LL = row.get('LL', np.nan)
    
    # Check if all required values are present
    if pd.isna(Wn) or pd.isna(PL) or pd.isna(LL):
        return np.nan
    
    # Check for division by zero (LL == PL means non-plastic)
    PI = LL - PL
    if PI == 0:
        return np.nan
    
    LI = (Wn - PL) / PI
    return LI

# Calculate LI for all records
df['Liquidity_Index'] = df.apply(calculate_liquidity_index, axis=1)

# Report statistics
li_available = df['Liquidity_Index'].notna().sum()
print(f"\n{'='*60}")
print("LIQUIDITY INDEX CALCULATION")
print(f"{'='*60}")
print(f"Records with LI: {li_available} / {len(df)} ({100*li_available/len(df):.1f}%)")

if li_available > 0:
    print(f"\nLiquidity Index Statistics:")
    print(df['Liquidity_Index'].describe().round(3))
    
    # Count state distributions
    brittle = (df['Liquidity_Index'] < 0).sum()
    plastic = ((df['Liquidity_Index'] >= 0) & (df['Liquidity_Index'] <= 1)).sum()
    liquid = (df['Liquidity_Index'] > 1).sum()
    
    print(f"\nSoil State Distribution:")
    print(f"  Brittle (LI < 0):     {brittle:5d} samples")
    print(f"  Plastic (0 ≤ LI ≤ 1): {plastic:5d} samples")
    print(f"  Liquid (LI > 1):      {liquid:5d} samples")

print(f"{'='*60}\n")

In [ ]:
# Visualize Liquidity Index distribution
if df['Liquidity_Index'].notna().sum() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    df['Liquidity_Index'].dropna().hist(bins=40, ax=axes[0], 
                                         color='teal', edgecolor='black', alpha=0.7)
    axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Plastic Limit')
    axes[0].axvline(1, color='orange', linestyle='--', linewidth=2, label='Liquid Limit')
    axes[0].set_xlabel('Liquidity Index', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Distribution of Liquidity Index', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Box plot
    df.boxplot(column='Liquidity_Index', ax=axes[1], vert=True, 
               patch_artist=True, boxprops=dict(facecolor='lightblue'))
    axes[1].set_ylabel('Liquidity Index', fontsize=12)
    axes[1].set_title('LI Variability', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
## 3. Synthetic CPT Resistance from SPT

### Empirical Correlations:
- **Cohesive soils:** $q_c \approx 0.4 \times N$
- **Granular soils:** $q_c \approx 0.6 \times N$

**Classification Logic:**
- Use `Fines_Content` to distinguish soil types:
  - Fines > 50% → Cohesive (clay/silt)
  - Fines ≤ 50% → Granular (sand/gravel)

In [ ]:
def classify_soil_behavior(row):
    """
    Classify soil as cohesive or granular based on fines content.
    
    Returns:
        str: 'cohesive', 'granular', or 'unknown'
    """
    fines = row.get('Fines_Content', np.nan)
    
    if pd.isna(fines):
        # Fallback: use plasticity if available
        is_plastic = row.get('is_plastic', np.nan)
        if is_plastic == 1:
            return 'cohesive'
        elif is_plastic == 0:
            return 'granular'
        return 'unknown'
    
    if fines > 50:
        return 'cohesive'
    else:
        return 'granular'

# Classify all soils
df['Soil_Behavior'] = df.apply(classify_soil_behavior, axis=1)

print(f"\nSoil Behavior Classification:")
print(df['Soil_Behavior'].value_counts())

In [ ]:
def calculate_synthetic_cpt(row):
    """
    Calculate synthetic CPT resistance from SPT N-value.
    
    Correlations:
    - Cohesive: qc ≈ 0.4 × N (MPa) = 400 × N (kPa)
    - Granular: qc ≈ 0.6 × N (MPa) = 600 × N (kPa)
    
    Returns:
        float: Synthetic CPT qc in kPa, or NaN
    """
    # If CPT is already available, don't override it
    if pd.notna(row.get('CPT_qc', np.nan)):
        return row['CPT_qc']
    
    # Check if SPT is available
    spt_n = row.get('SPT_N', np.nan)
    if pd.isna(spt_n):
        return np.nan
    
    # Apply correlation based on soil behavior
    soil_type = row.get('Soil_Behavior', 'unknown')
    
    if soil_type == 'cohesive':
        return 400 * spt_n  # kPa
    elif soil_type == 'granular':
        return 600 * spt_n  # kPa
    else:
        # Default: use average correlation
        return 500 * spt_n  # kPa

# Calculate synthetic CPT
df['CPT_qc_synthetic'] = df.apply(calculate_synthetic_cpt, axis=1)

# Create a unified CPT column (original or synthetic)
df['CPT_qc_unified'] = df['CPT_qc'].fillna(df['CPT_qc_synthetic'])

# Report statistics
original_cpt = df['CPT_qc'].notna().sum()
synthetic_cpt = df['CPT_qc_synthetic'].notna().sum() - original_cpt
total_cpt = df['CPT_qc_unified'].notna().sum()

print(f"\n{'='*60}")
print("SYNTHETIC CPT RESISTANCE CALCULATION")
print(f"{'='*60}")
print(f"Original CPT measurements:  {original_cpt:5d}")
print(f"Synthetic CPT (from SPT):   {synthetic_cpt:5d}")
print(f"Total CPT availability:     {total_cpt:5d} / {len(df)} ({100*total_cpt/len(df):.1f}%)")
print(f"{'='*60}\n")

---
## 4. Additional Derived Features

In [ ]:
# Activity Ratio (for clays): PI / Clay_Fraction
# Useful for identifying expansive clays
def calculate_activity(row):
    """Activity = PI / (% passing 2 microns)"""
    PI = row.get('PI', np.nan)
    fines = row.get('Fines_Content', np.nan)
    
    if pd.isna(PI) or pd.isna(fines) or fines == 0:
        return np.nan
    
    # Approximate clay fraction as 0.6 * fines for typical soils
    clay_fraction = 0.6 * fines
    
    if clay_fraction == 0:
        return np.nan
    
    return PI / clay_fraction

df['Activity_Ratio'] = df.apply(calculate_activity, axis=1)

# Consistency Index (CI): Opposite of LI
# CI = (LL - Wn) / (LL - PL)
# Higher CI = Stiffer soil
df['Consistency_Index'] = 1 - df['Liquidity_Index']

# Relative Density proxy for sands: SPT normalized by effective stress
# This is a simplified version; true calculation requires stress state
if 'SPT_N' in df.columns and 'Depth' in df.columns:
    # Assume σ'v ≈ 18 * Depth (kPa) for typical soils
    df['SPT_N_normalized'] = df['SPT_N'] / np.sqrt(df['Depth'] * 18 / 100)

print("\nDerived Features Created:")
print(f"  - Liquidity_Index")
print(f"  - Consistency_Index")
print(f"  - Activity_Ratio")
print(f"  - CPT_qc_synthetic")
print(f"  - CPT_qc_unified")
print(f"  - Soil_Behavior")
if 'SPT_N_normalized' in df.columns:
    print(f"  - SPT_N_normalized")

---
## 5. Feature Correlation Analysis

Examine relationships between engineered features and target variables.

In [ ]:
# Select features for correlation analysis
feature_cols = ['LL', 'PL', 'PI', 'Moisture_Content', 'Fines_Content', 
                'Unit_Weight', 'SPT_N', 'CPT_qc_unified', 'Liquidity_Index',
                'Consistency_Index', 'Activity_Ratio']

target_cols = ['Angle_Internal_Friction', 'Undrained_Cohesion']

# Filter to available columns
feature_cols = [c for c in feature_cols if c in df.columns]
target_cols = [c for c in target_cols if c in df.columns]

if target_cols:
    # Create correlation matrix
    corr_cols = feature_cols + target_cols
    corr_matrix = df[corr_cols].corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, linewidths=0.5, cbar_kws={'label': 'Correlation'})
    plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Print correlations with targets
    print(f"\n{'='*60}")
    print("FEATURE CORRELATIONS WITH TARGETS")
    print(f"{'='*60}")
    for target in target_cols:
        print(f"\n{target}:")
        target_corr = corr_matrix[target].drop(target_cols).sort_values(ascending=False)
        for feat, corr in target_corr.items():
            if abs(corr) > 0.1:  # Only show meaningful correlations
                print(f"  {feat:25s}: {corr:6.3f}")
    print(f"{'='*60}\n")

---
## 6. Export Engineered Dataset

In [ ]:
# Final feature summary
print(f"\n{'='*60}")
print("FEATURE ENGINEERING SUMMARY")
print(f"{'='*60}")
print(f"Total Records:  {len(df)}")
print(f"Total Features: {len(df.columns)}")
print(f"\nNew Features Added:")
new_features = ['Liquidity_Index', 'Consistency_Index', 'Activity_Ratio',
                'CPT_qc_synthetic', 'CPT_qc_unified', 'Soil_Behavior']
if 'SPT_N_normalized' in df.columns:
    new_features.append('SPT_N_normalized')

for feat in new_features:
    if feat in df.columns:
        count = df[feat].notna().sum()
        pct = 100 * count / len(df)
        print(f"  {feat:25s}: {count:5d} / {len(df)} ({pct:5.1f}%)")

print(f"{'='*60}\n")

In [ ]:
# Export engineered dataset
output_path = '/home/claude/geotechnical_engineered.csv'
df.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"✓ ENGINEERED DATASET EXPORTED")
print(f"{'='*60}")
print(f"Location: {output_path}")
print(f"Shape:    {df.shape}")
print(f"\nReady for Comparative Modeling (Phase 4)")
print(f"{'='*60}\n")

---
## Summary

**Feature Engineering Complete:**
1. ✓ Liquidity Index (LI) - Soil consistency indicator
2. ✓ Synthetic CPT resistance from SPT correlations
3. ✓ Soil behavior classification
4. ✓ Additional derived indices (Activity, Consistency)
5. ✓ Correlation analysis with targets

**Next Phase:** Comparative_Analysis.ipynb (ML Modeling)